# 雪乃 A100 · 原作章节问答续训 v4

从 20 本 EPUB 的章节和带编号的段落出发，由 **Qwen2.5-14B-Instruct**
在同一台 A100 上生成剧情、人物关系、前因后果问答并自动复核。
每条录用问答有原文段落编号和可核对的短证据；自动核验结果仍需人工抽查。
问答整理完成后，用它们和既有人格样本继续微调 Qwen2.5-1.5B 的 LoRA。
源码、章节数据、问答、日志、训练断点和最终 LoRA 都保存在 Drive。

先运行 **1 → 2 → 3 → 3.1**。第 3.1 节可能跨多个 Colab 会话；重连后按原顺序重跑，
会保留已完成的片段，进度见 `teacher/manifest.json`。全部通过后运行 **4** 续训，
**5** 对比训练前后，**6** 可选导出 GGUF，**7** 可手动并排提问。


## 1. 检查 A100，挂载云盘

在「代码执行程序 → 更改运行时类型」选择 A100 GPU，保存并连接。
续跑保持 `RUN_ID`、起始适配器和训练参数一致。`ATTEMPT` 仅用于首次保存前中断、没有可恢复断点时。
默认使用已经完成的 v1 人格适配器；如 v3 已完成，可在第一次运行前把 `INITIAL_RUN`、`INITIAL_STAGE` 和 `INITIAL_TASK` 改为对应值。


In [ ]:
import json, os, re, shutil, subprocess, sys, tempfile
from pathlib import Path
from hashlib import sha256
from zipfile import ZipFile
from google.colab import drive

# 实际 CUDA 设备决定能否训练；笔记本名称或元数据不会分配 GPU。
def require_a100():
    import shutil
    import subprocess
    from pathlib import Path

    help_text = ("请打开「代码执行程序（Runtime）→ 更改运行时类型 → A100 GPU」，"
                 "保存并重新连接，然后从第 1 节运行。只写 A100 配置不会自动获得 GPU。")
    smi = shutil.which("nvidia-smi") or next(
        (p for p in ("/usr/bin/nvidia-smi", "/usr/local/nvidia/bin/nvidia-smi") if Path(p).is_file()), None)
    if smi:
        try:
            print(subprocess.check_output([smi, "--query-gpu=name,memory.total", "--format=csv,noheader"],
                                          text=True, stderr=subprocess.STDOUT, timeout=15).strip())
        except (OSError, subprocess.SubprocessError) as error:
            print("nvidia-smi 诊断不可用，将使用 PyTorch 检查实际 GPU：", type(error).__name__)
    else:
        print("未找到 nvidia-smi，正在检查 PyTorch 是否能访问 CUDA GPU。")
    try:
        import torch
        if not torch.cuda.is_available():
            raise SystemExit("当前没有可用的 CUDA GPU，训练尚未开始。" + help_text)
        name = torch.cuda.get_device_name(0)
        if "A100" not in name.upper():
            raise SystemExit(f"当前实际分配的是 {name}，本笔记本要求 A100。" + help_text)
        if not torch.cuda.is_bf16_supported():
            raise SystemExit("当前 CUDA 环境不支持本训练所需的 BF16。" + help_text)
        print("CUDA 检查通过：", name)
        return name
    except Exception as error:
        raise SystemExit("无法初始化当前运行时的 CUDA/PyTorch：" + str(error) + "。" + help_text) from None

gpu = require_a100()
drive.mount("/content/drive")
RUN_ID = "epub-a100-v4-chapters"
MAX_EPOCHS = 6
EARLY_STOP_PATIENCE = 3
ATTEMPT = 1
INITIAL_RUN = "epub-a100-v1"
INITIAL_STAGE = "persona-01"
INITIAL_TASK = "complete_persona"  # v3 已完成时可选 distill_sft
for value in (RUN_ID, INITIAL_RUN, INITIAL_STAGE):
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", value):
        raise ValueError("实验和阶段名只能使用英文字母、数字、横线和下划线。")
if type(MAX_EPOCHS) is not int or not 2 <= MAX_EPOCHS <= 50 or type(EARLY_STOP_PATIENCE) is not int or not 1 <= EARLY_STOP_PATIENCE <= MAX_EPOCHS:
    raise ValueError("轮次范围 2–50，早停耐心值必须在 1 到总轮次之间。")
if type(ATTEMPT) is not int or ATTEMPT < 1:
    raise ValueError("ATTEMPT 必须为正整数。")
project = Path("/content/drive/MyDrive/Yukino-Companion")
if not project.is_dir():
    raise FileNotFoundError("未找到已上传的 Yukino-Companion 云盘目录。")
root = Path("/content/yukino-chapter-v4-runtime")
root.mkdir(exist_ok=True)
drive_runs = project / "retrain" / RUN_ID
drive_runs.mkdir(parents=True, exist_ok=True)
epubs = project / "data/epubs"
if not list(epubs.glob("*.epub")):
    raise FileNotFoundError(epubs)
initial_source = project / "retrain" / INITIAL_RUN / INITIAL_STAGE
persona_output = drive_runs / f"distill-{ATTEMPT:02d}"
base_revision = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"

def digest(path):
    h = sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024**2), b""):
            h.update(block)
    return h.hexdigest()

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    temporary.replace(path)

def inventory(directory):
    return {p.relative_to(directory).as_posix(): digest(p) for p in sorted(Path(directory).rglob("*")) if p.is_file() and p.name != "artifacts.json"}

def publish_directory(local, final):
    final.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix=".publish-", dir=final.parent) as staging:
        candidate = Path(staging) / final.name
        shutil.copytree(local, candidate)
        if inventory(candidate) != inventory(local):
            raise RuntimeError("复制后的 Drive 文件校验失败。")
        candidate.rename(final)

def verify_run(output, task):
    report = load_json(output / "training_report.json")
    if not (report.get("completed") and report.get("run_purpose") == "training"
            and report.get("task") == task and report.get("base_revision") == base_revision
            and report.get("device") == "cuda" and report.get("dtype") == "torch.bfloat16"
            and report.get("steps", 0) > 1
            and report.get("adapter_sha256") == digest(output / "adapter/adapter_model.safetensors")):
        raise RuntimeError(f"模型报告或权重校验失败：{output}")
    return report

initial_report = verify_run(initial_source, INITIAL_TASK)
print("继续训练来源", initial_source, initial_report["adapter_sha256"])
print("持久化输出", drive_runs)


## 2. 恢复代码和训练底座

从 Drive 校验并解包固定源码。保留 Colab 自带 CUDA Torch，安装其余固定版本依赖。
隔离环境跳过 Python 自带的 pip 初始化，使用 Colab 的 pip 安装依赖，安装日志保存在云盘 `environment.log`。
1.5B 学生底座和最终 LoRA 保存在 Drive。14B 教师的公开权重可随时重新下载，默认用 Colab 本地缓存；原文和问答不提交给在线模型 API。


In [ ]:
import tomllib
source_archive = project / "source/training-source-chapter-v4.zip"
source_hash = "034ae00d84037a9baf54f7b46e25d53ba421be094fa18ec44a1e384e6ac31b13"
if not source_archive.is_file() or digest(source_archive) != source_hash:
    raise RuntimeError(f"源码归档缺失或 SHA 不符：{source_archive}")
repo = project / "source" / ("chapter-v4-" + source_hash[:12])
with ZipFile(source_archive) as archive:
    for item in archive.infolist():
        if item.filename.startswith("/") or ".." in Path(item.filename).parts:
            raise ValueError("源码归档存在不安全路径。")
    if not repo.exists():
        with tempfile.TemporaryDirectory(prefix=".source-", dir=repo.parent) as stage:
            candidate = Path(stage) / "source"
            archive.extractall(candidate)
            candidate.rename(repo)
    for item in archive.infolist():
        if not item.is_dir() and (repo / item.filename).read_bytes() != archive.read(item):
            raise RuntimeError("云盘源码快照被改动：" + item.filename)
service = repo / "model-service"
venv = root / "venv-no-ensurepip"
python = venv / "bin/python"
def run(*args, cwd=service, log=None, interpreter=None):
    command = [str(interpreter or python), *(str(a) for a in args)]
    env = dict(os.environ, USE_TF="0", USE_TORCH="1", PYTHONPATH=str(service), PYTHONUNBUFFERED="1")
    log = Path(log or drive_runs / "operations.log")
    with log.open("a", encoding="utf-8", buffering=1) as stream:
        with subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding="utf-8", errors="replace", bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                stream.write(line)
            if process.wait():
                raise subprocess.CalledProcessError(process.returncode, command)
env_log = drive_runs / "environment.log"
ready = venv / "deps-ready"
if not python.is_file():
    run("-m", "venv", "--system-site-packages", "--without-pip", venv,
        interpreter=sys.executable, log=env_log)
cuda_check = ("import torch; print(torch.__version__, torch.version.cuda, torch.__file__); "
              "assert tuple(map(int,torch.__version__.split('.')[:2])) >= (2,6); "
              "assert torch.cuda.is_available() and 'A100' in torch.cuda.get_device_name(0) "
              "and torch.cuda.is_bf16_supported(); print(torch.ones(1,device='cuda',dtype=torch.bfloat16))")
run("-c", cuda_check, log=env_log)
config = tomllib.loads((service / "pyproject.toml").read_text(encoding="utf-8"))["project"]
dependencies = config["dependencies"] + [s for s in config["optional-dependencies"]["train"] if not s.startswith("torch")]
if not ready.exists():
    from importlib.metadata import version
    constraints = root / "colab-torch-constraint.txt"
    constraints.write_text("torch==" + version("torch") + "\n", encoding="utf-8")
    run("-m", "pip", "--python", python, "install", "--disable-pip-version-check",
        "--retries", "3", "--timeout", "60", "--constraint", constraints, *dependencies,
        interpreter=sys.executable, log=env_log)
    run("-c", cuda_check, log=env_log)
run("-c", "from transformers import Trainer; from peft import PeftModel; from yuki_model.distillation import build_distillation", log=env_log)
ready.touch()
run("-m", "pip", "--python", python, "list", "--format=json", interpreter=sys.executable, log=env_log)
base = project / "models/qwen2.5-1.5b-instruct-989aa798"
if not (base / "BASE_PROVENANCE.json").is_file():
    old_cache = Path("/content/yukino-a100/source/model-service/models/base")
    if not base.exists() and (old_cache / "BASE_PROVENANCE.json").is_file():
        publish_directory(old_cache, base)
    else:
        run("-m", "yuki_model", "download", "--output", base)
base_proof = load_json(base / "BASE_PROVENANCE.json")
if base_proof["base_revision"] != base_revision:
    raise RuntimeError("基座版本不匹配。")
for name, expected in base_proof["files"].items():
    if Path(name).name != name or digest(base / name) != expected:
        raise RuntimeError("基座文件校验失败：" + name)
if digest(base / "model.safetensors") != "dd924a11b4c220f385b51ffa522daea7c9f3d850e31b162bb5661df483c6d3ee":
    raise RuntimeError("基座权重不符合固定版本。")
print("代码路径", repo)
print("基座路径", base)


## 3. 验证章节、原文来源与续训起点

逐卷核对 EPUB 的 SHA-256，加载预先整理的章节与片段。此步只整理和验证，不生成问答。
输出卷数、章数、字符数和仍需补录的覆盖缺口；段落编号用于之后回查证据。


In [ ]:
inputs = {
    "source_archive_sha256": source_hash, "base_revision": base_revision,
    "initial_path": str(initial_source), "initial_sha256": initial_report["adapter_sha256"],
    "initial_config_sha256": digest(initial_source / "adapter/adapter_config.json"),
    "corpus_archive_sha256": "b29064ac13e5153b9bae250fc26e1aed0dff378324598130515a578ebf4e8115", "initial_task": INITIAL_TASK,
    "epochs": MAX_EPOCHS, "patience": EARLY_STOP_PATIENCE,
    "epubs": {p.name: digest(p) for p in sorted(epubs.glob("*.epub"))},
}
input_file = drive_runs / "inputs.json"
if input_file.exists() and load_json(input_file) != inputs:
    raise RuntimeError("该 RUN_ID 已绑定另一份输入或参数，请换新 RUN_ID。")
save_json(input_file, inputs)
initial_snapshot = drive_runs / "initial"
if not initial_snapshot.exists():
    with tempfile.TemporaryDirectory(prefix=".initial-", dir=drive_runs) as stage:
        candidate = Path(stage) / "initial"
        candidate.mkdir()
        shutil.copytree(initial_source / "adapter", candidate / "adapter")
        shutil.copyfile(initial_source / "training_report.json", candidate / "training_report.json")
        if digest(candidate / "adapter/adapter_model.safetensors") != inputs["initial_sha256"]:
            raise RuntimeError("旧权重快照复制校验失败。")
        candidate.rename(initial_snapshot)
verify_run(initial_snapshot, INITIAL_TASK)
if digest(initial_snapshot / "adapter/adapter_config.json") != inputs["initial_config_sha256"]:
    raise RuntimeError("旧适配器配置与快照不符。")

corpus_archive = project / "data/chapter-corpus-v4.zip"
if digest(corpus_archive) != inputs["corpus_archive_sha256"]:
    raise RuntimeError("章节档案的 SHA-256 不符。")
corpus = drive_runs / "corpus"
expected_files = {"manifest.json", "chapters.jsonl", "passages.jsonl", "vol11-interlude-supplement.jsonl"}
with ZipFile(corpus_archive) as archive:
    if set(archive.namelist()) != expected_files or len(archive.namelist()) != len(expected_files):
        raise RuntimeError("章节档案的文件列表不符。")
    if not corpus.exists():
        with tempfile.TemporaryDirectory(prefix=".corpus-", dir=drive_runs) as stage:
            candidate = Path(stage) / "data"
            archive.extractall(candidate)
            candidate.rename(corpus)
    for name in expected_files:
        if digest(corpus / name) != sha256(archive.read(name)).hexdigest():
            raise RuntimeError("章节档案与运行目录不一致：" + name)
corpus_manifest = load_json(corpus / "manifest.json")
if digest(corpus / "vol11-interlude-supplement.jsonl") != corpus_manifest["supplement_sha256"]:
    raise RuntimeError("图片插叙补录的 SHA-256 不符。")
if corpus_manifest["book_count"] != 20 or {b["book"]: b["source_sha256"] for b in corpus_manifest["books"]} != inputs["epubs"]:
    raise RuntimeError("云盘 EPUB 与整理章节时使用的 20 本原作不一致。")
print(json.dumps({k: v for k, v in corpus_manifest.items() if k not in {"books", "files", "artifacts"}}, ensure_ascii=False, indent=2))
prepared = drive_runs / "prepared"


## 3.1. 在 A100 上逐段生成和核验问答

教师逐段读取带编号的正文，提问并核对答案能否由证据支持，依次写入 `teacher/jobs/`。
可设置 `PASSAGES_THIS_SESSION = 30` 分批完成；每次重新运行此节继续剩余片段。
首次下载 14B 权重与生成都会花时间。预计时长以实测进度为准。
首轮全部片段已有记录时，重跑本节会校验已有记录并用短输出格式补跑失败片段。
默认每次并行生成 2 个片段并批量复核。显存不足会降到单条，已完成记录继续复用。
本次恢复的总量下限为 850 条有效原文问答，验证集和测试集各至少 30 条，其余核验和覆盖要求沿用原设置。
所有片段处理完并达到覆盖门槛后才会生成训练文件。


In [ ]:
if "stop_comparison_worker" in globals():
    stop_comparison_worker()
PASSAGES_THIS_SESSION = None  # None = 处理所有剩余片段，整数 = 本次最多处理多少片段
RECOVERY_BATCH_SIZE = 2
CACHE_TEACHER_ON_DRIVE = False  # 若要保留 ~30 GB 的公开教师权重，可改为 True
original_teacher = drive_runs / "teacher"
first_pass_manifest = original_teacher / "manifest.json"
first_pass = load_json(first_pass_manifest) if first_pass_manifest.is_file() else {}
recover_failed = (not first_pass.get("complete") and first_pass.get("passages_total", 0) > 0
                  and first_pass.get("passages_completed", 0) + first_pass.get("passages_failed", 0)
                  == first_pass["passages_total"])
allow_unfinalized = (not first_pass and (original_teacher / "settings.json").is_file()
                    and sum(1 for _ in (original_teacher / "jobs").glob("*.json"))
                    == load_json(corpus / "manifest.json")["passage_count"])
recover_failed = recover_failed or allow_unfinalized
teacher = drive_runs / "teacher-recovered" if recover_failed else original_teacher
teacher_manifest = teacher / "manifest.json"
if not teacher_manifest.is_file() or not load_json(teacher_manifest).get("complete"):
    teacher_cache = (project / "models" if CACHE_TEACHER_ON_DRIVE else root) / "teacher-hf"
    download = '''
import sys
from huggingface_hub import snapshot_download
print(snapshot_download(repo_id="Qwen/Qwen2.5-14B-Instruct",
      revision="cf98f3b3bbb457ad9e2bb7baf9a0125b6b88caa8", cache_dir=sys.argv[1],
      allow_patterns=["*.json", "*.safetensors", "*.txt", "*.jinja", "LICENSE"], max_workers=4))
'''
    teacher_weights = (teacher_cache / "models--Qwen--Qwen2.5-14B-Instruct/snapshots/"
                       "cf98f3b3bbb457ad9e2bb7baf9a0125b6b88caa8")
    if not (teacher_weights / "model.safetensors.index.json").is_file() or not all(
        (teacher_weights / name).is_file() for name in set(load_json(teacher_weights / "model.safetensors.index.json")["weight_map"].values())
    ):
        run("-c", download, teacher_cache, log=drive_runs / "teacher-download.log")
    args = [] if PASSAGES_THIS_SESSION is None else ["--limit", str(PASSAGES_THIS_SESSION)]
    if recover_failed:
        recovery_script = project / "source/recover_chapter_teacher_batched.py"
        if digest(recovery_script) != "98a5e37b6fc4655e31df748ed35a1f4ee146e421394cc1894e1dad056206ea05":
            raise RuntimeError("恢复脚本的 SHA-256 不一致。")
        if allow_unfinalized:
            args += ["--allow-unfinalized"]
        run(recovery_script, "--passages", corpus / "passages.jsonl", "--original", original_teacher,
            "--model", teacher_weights, "--output", teacher, "--batch-size", RECOVERY_BATCH_SIZE,
            *args, log=drive_runs / "teacher-recovery.log")
    else:
        run("-m", "yuki_model.chapter_teacher", "--passages", corpus / "passages.jsonl",
            "--model", teacher_weights, "--output", teacher, *args, log=drive_runs / "teacher.log")
if not teacher_manifest.is_file() or not load_json(teacher_manifest).get("complete"):
    raise SystemExit("生成进度已保存。请重新运行第 3.1 节，完成后再运行第 4 节。")

replay = drive_runs / "rehearsal-v3"
if not replay.exists():
    with tempfile.TemporaryDirectory(prefix=".rehearsal-", dir=drive_runs) as stage:
        candidate = Path(stage) / "data"
        run("scripts/build_distillation_sft.py", "--epubs", epubs, "--output", candidate,
            "--tokenizer", base, "--sequence-length", "1536", log=drive_runs / "rehearsal.log")
        candidate.rename(replay)
if not prepared.exists():
    with tempfile.TemporaryDirectory(prefix=".assemble-", dir=drive_runs) as stage:
        candidate = Path(stage) / "data"
        run("-m", "yuki_model.chapter_dataset", "--corpus", corpus, "--teacher", teacher,
            "--replay", replay, "--character-pack", repo / "characters/yukino",
            "--tokenizer", base, "--output", candidate, "--sequence-length", "1536",
            "--min-source-rows", "850",
            log=drive_runs / "assemble.log")
        candidate.rename(prepared)
data_manifest = load_json(prepared / "manifest.json")
print("实际训练、验证、测试样本数：", data_manifest["examples"])
print("数据来源：", data_manifest.get("origin_counts"))


## 4. 用 A100 续训，按轮保存断点

第 3.1 节组装完成后才开始。最多 6 轮，保留每轮完整断点并按验证集选择最佳一轮。
第一次训练按最长样本选择每批 4 条或 8 条。每次权重更新仍使用约 8 条样本，断点续跑沿用已保存的批量设置。
更换会话后同样从第 1 节开始；参数相同会读取已有生成进度，并从最近完整训练断点继续。
核验云盘中的最终适配器后，自动生成 15 轮对照报告，再同步云盘并释放当前 Colab 运行时。
如需接着运行第 5～7 节，可以重新连接，或在第 4 节将 `RELEASE_AFTER_STAGE4` 设为 `False`。


In [ ]:
if not (drive_runs / "prepared/manifest.json").is_file():
    raise SystemExit("请先完成第 3.1 节的生成和组装。")
if "stop_comparison_worker" in globals():
    stop_comparison_worker()
config = load_json(service / "configs/train-distill.json")
config.update(model=str(base), initial_adapter=str(initial_snapshot / "adapter"),
              train_file=str(prepared / "train.jsonl"), eval_file=str(prepared / "val.jsonl"),
              data_manifest=str(prepared / "manifest.json"), output_dir=str(persona_output),
              epochs=MAX_EPOCHS, save_total_limit=MAX_EPOCHS,
              early_stopping_patience=EARLY_STOP_PATIENCE)
config_file = drive_runs / f"{persona_output.name}-config.json"
A100_BATCH_SIZE = None  # 按样本长度选择，也可在首次训练前手动设为 2 或 4。
if config_file.exists():
    saved_config = load_json(config_file)
    config.update(batch_size=saved_config["batch_size"],
                  gradient_accumulation_steps=saved_config["gradient_accumulation_steps"])
else:
    longest = load_json(prepared / "manifest.json")["max_tokens"]
    batch = (8 if 0 < longest <= 768 else 4) if A100_BATCH_SIZE is None else A100_BATCH_SIZE
    if batch not in (2, 4, 8):
        raise ValueError("A100_BATCH_SIZE 请选择 None、2、4 或 8。")
    config.update(batch_size=batch, gradient_accumulation_steps=8 // batch)
print("A100 每批样本数", config["batch_size"], "梯度累积次数", config["gradient_accumulation_steps"])
if config_file.exists() and load_json(config_file) != config:
    raise RuntimeError("已有训练参数不同，请使用新的 RUN_ID。")
save_json(config_file, config)
if (persona_output / "training_report.json").is_file():
    report = verify_run(persona_output, "distill_sft")
    if report["data"]["data_manifest_sha256"] != digest(prepared / "manifest.json"):
        raise RuntimeError("已完成模型与当前数据不一致。")
    print("此阶段已经完成，读取保存结果。")
else:
    resume = persona_output.exists() and any(persona_output.iterdir())
    runner = '''
import json, sys
from pathlib import Path
from yuki_model.training import train, validate_resume, validate_training_spec
from yuki_model.io import file_sha
c = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
if sys.argv[2] == "resume":
    c["resume"] = True
    try:
        selected = validate_resume(Path(c["output_dir"]), c, {
            "train_sha256": file_sha(c["train_file"]), "eval_sha256": file_sha(c["eval_file"]),
            **validate_training_spec(c)})
    except (ValueError, OSError) as error:
        raise RuntimeError("没有可恢复的完整断点；保留原目录，增加第1节 ATTEMPT 后重跑。详情：" + str(error)) from error
    print("恢复断点", selected, flush=True)
print(json.dumps(train(c), ensure_ascii=False, indent=2))
'''
    run("-c", runner, config_file, "resume" if resume else "fresh", log=drive_runs / f"{persona_output.name}.log")
    report = verify_run(persona_output, "distill_sft")
print("实际完成轮次", report["actual_epoch"], "上限", report["configured_epochs"], "早停", report["stopped_early"])
print("导出权重来源", report["selected_weights"], report["best_checkpoint"])
print("训练前/后验证损失", (report["before"] or {}).get("eval_loss"), report["after"]["eval_loss"])
print("模型已保存在 Drive", persona_output / "adapter")


RELEASE_AFTER_STAGE4 = True
MULTI_TURN_AFTER_STAGE4 = True
verified = verify_run(persona_output, "distill_sft")
final_status = {
    "training_completed": True,
    "adapter_path": str(persona_output / "adapter"),
    "adapter_sha256": verified["adapter_sha256"],
    "multi_turn": "disabled",
    "release_after_flush": RELEASE_AFTER_STAGE4,
}
if MULTI_TURN_AFTER_STAGE4:
    eval_archive = project / "source/yukino-v4-multi-turn-eval.zip"
    eval_output = drive_runs / "evaluation" / persona_output.name / "multi-turn"
    final_status["multi_turn_report"] = str(eval_output / "report.json")
    try:
        if digest(eval_archive) != "8ef48d3fafe37f84f013a2061fcd4405290dda796f6c0560a862a0b21e8b69ad":
            raise RuntimeError("多轮测试包的 SHA-256 不一致。")
        with tempfile.TemporaryDirectory(prefix="multi-turn-eval-", dir=root) as temp:
            tools_dir = Path(temp)
            with ZipFile(eval_archive) as archive:
                if sorted(archive.namelist()) != ["cases.json", "evaluate_distillation.py", "evaluate_multi_turn.py"]:
                    raise RuntimeError("多轮测试包的文件列表不一致。")
                archive.extractall(tools_dir)
            run(tools_dir / "evaluate_multi_turn.py", "--cases", tools_dir / "cases.json",
                "--character-pack", repo / "characters/yukino", "--model", base,
                "--before-adapter", initial_snapshot / "adapter",
                "--after-adapter", persona_output / "adapter", "--device", "cuda",
                "--output", eval_output, log=drive_runs / "multi-turn.log")
        evaluation = load_json(eval_output / "report.json")
        if not evaluation.get("generation_completed") or evaluation.get("completed_turn_pairs") != 15:
            raise RuntimeError("多轮评测没有生成全部 15 组回答。")
        if (evaluation["generation_sha256"] != digest(eval_output / "generations.jsonl")
                or evaluation["configuration_sha256"] != digest(eval_output / "evaluation_config.json")
                or (eval_output / "comparison.html").stat().st_size == 0):
            raise RuntimeError("多轮评测文件校验失败。")
        final_status["multi_turn"] = "completed"
    except Exception as error:
        final_status["multi_turn"] = "failed"
        final_status["multi_turn_error"] = str(error)
        print("多轮测试中断。模型已保存在云盘，详见 multi-turn.log。", flush=True)
save_json(persona_output / "finish-status.json", final_status)
if RELEASE_AFTER_STAGE4:
    from google.colab import drive, runtime
    print("正在同步已验证的模型、断点和报告，然后释放当前 Colab 运行时。", flush=True)
    drive.flush_and_unmount(timeout_ms=300_000)
    runtime.unassign()


## 5. 在保留章节上对比续训前后

用同一基座和角色提示生成两份答案。新生成的第 5 卷问答用于验证，第 10 卷问答只用于最终测试。
旧模型或复习样本可能已经接触过其中某些事实；原文证据可在数据文件中回查。
人格回归题另外出报告。请抽查答错、猜错因果、人物混淆的案例。


In [ ]:
from IPython.display import HTML, display
if "stop_comparison_worker" in globals():
    stop_comparison_worker()
verify_run(persona_output, "distill_sft")
for label, name in (("regression", "regression"), ("held-out", "test")):
    output = drive_runs / "evaluation" / persona_output.name / label
    run("scripts/evaluate_distillation.py", "--model", base, "--device", "cuda",
        "--before-adapter", initial_snapshot / "adapter", "--after-adapter", persona_output / "adapter",
        "--prompts", prepared / f"{name}-prompts.jsonl", "--references", prepared / f"{name}.jsonl",
        "--evaluation-kind", label, "--output", output, log=drive_runs / "evaluation.log")
    display(HTML((output / "comparison.html").read_text(encoding="utf-8")))
    print("云盘报告", output)


## 6. 导出候选离线 GGUF 模型

导出最佳验证轮次的 LoRA 为 Q4_K_M；合并 FP16 及转换中间文件也放在 Drive，
成功后仅清理本节创建的临时中间目录，最终 GGUF、来源报告与训练断点仍保留。
这是候选模型，文件完整性验证不代表回答质量合格。

In [ ]:
if "stop_comparison_worker" in globals():
    stop_comparison_worker()
# 5. 在本地合并与量化，再将校验后的候选 GGUF 保存到 Drive
EXPORT_GGUF = True  # 仅训练 LoRA 时可关闭

if EXPORT_GGUF:
    export_report = verify_run(persona_output, "distill_sft")
    export_path = drive_runs / f"{persona_output.name}-gguf"
    converter = root / "llama.cpp"
    converter_commit = "b3fa69955ea81c3b60e77e3c1e2224c4a826ffb9"

    if not export_path.exists():
        if shutil.disk_usage(drive_runs).free < 12 * 1024**3:
            raise RuntimeError("GGUF 合并与量化至少需要 12 GiB 云盘可用空间。")
        if not (converter / ".git").is_dir():
            subprocess.run(["git", "init", str(converter)], check=True)
        subprocess.run(["git", "-C", str(converter), "fetch", "--depth=1",
                        "https://github.com/ggml-org/llama.cpp.git", converter_commit], check=True)
        subprocess.run(["git", "-C", str(converter), "checkout", "--detach", "FETCH_HEAD"], check=True)
        actual = subprocess.check_output(["git", "-C", str(converter), "rev-parse", "HEAD"], text=True).strip()
        if actual != converter_commit:
            raise RuntimeError("llama.cpp 转换器版本不匹配。")
        run("-m", "pip", "install", "numpy==2.2.6", "sentencepiece==0.2.1", "protobuf==4.25.8")
        build = converter / "build"
        subprocess.run(["cmake", "-S", str(converter), "-B", str(build),
                        "-DCMAKE_BUILD_TYPE=Release", "-DGGML_NATIVE=OFF",
                        "-DLLAMA_BUILD_TESTS=OFF", "-DLLAMA_BUILD_TOOLS=ON",
                        "-DLLAMA_BUILD_COMMON=ON", "-DLLAMA_BUILD_APP=OFF",
                        "-DLLAMA_BUILD_SERVER=OFF", "-DLLAMA_BUILD_EXAMPLES=OFF",
                        "-DLLAMA_OPENSSL=OFF"], check=True)
        subprocess.run(["cmake", "--build", str(build), "--target", "llama-quantize", "--parallel", "2"], check=True)
        with tempfile.TemporaryDirectory(prefix=".export-", dir=drive_runs) as work:
            work = Path(work)
            merged = work / "merged"
            candidate = work / "gguf"
            run("-m", "yuki_model", "merge", "--base", base, "--adapter", persona_output / "adapter",
                "--output", merged)
            merge_check = """
import sys
from safetensors import safe_open
import torch

key = "model.layers.0.self_attn.q_proj.weight"
with safe_open(sys.argv[1], framework="pt", device="cpu") as file:
    original = file.get_tensor(key)
with safe_open(sys.argv[2], framework="pt", device="cpu") as file:
    updated = file.get_tensor(key)
if original.shape != updated.shape or not torch.isfinite(updated).all():
    raise RuntimeError("Merged tensor is incompatible or non-finite")
changed = int(torch.count_nonzero(updated != original.to(updated.dtype)).item())
if changed == 0:
    raise RuntimeError("LoRA merge did not change the targeted tensor")
print(changed)
"""
            checked = subprocess.check_output(
                [str(python), "-c", merge_check, str(base / "model.safetensors"), str(merged / "model.safetensors")],
                text=True,
            ).strip()
            save_json(merged / "merge-validation.json", {
                "checkedTensor": "model.layers.0.self_attn.q_proj.weight",
                "changedElements": int(checked),
                "adapterSha256": export_report["adapter_sha256"],
            })
            run("scripts/export_gguf.py", "--llama-cpp", converter, "--merged", merged,
                "--training-report", persona_output / "training_report.json", "--base-license", base / "LICENSE",
                "--quantizer", build / "bin/llama-quantize", "--output", candidate,
                "--work-dir", work / "conversion", "--remove-fp16")
            manifest = load_json(candidate / "manifest.json")
            manifest["name"] = f"雪乃本地角色模型 · Qwen 1.5B · A100 · {RUN_ID}/{persona_output.name}"
            save_json(candidate / "manifest.json", manifest)
            publish_directory(candidate, export_path)

    manifest = load_json(export_path / "manifest.json")
    weights = export_path / manifest["file"]
    with weights.open("rb") as stream:
        if stream.read(4) != b"GGUF":
            raise RuntimeError("导出的文件缺少 GGUF 文件头。")
    if (manifest["training"]["adapterSha256"] != export_report["adapter_sha256"]
            or manifest["llamaCppConverterCommit"] != converter_commit
            or weights.stat().st_size != manifest["bytes"] or digest(weights) != manifest["sha256"]):
        raise RuntimeError("Drive 上的 GGUF 权重或来源校验失败。")
    print("候选 GGUF 已保存并通过文件校验，等待人工评估：", weights)
else:
    print("本次跳过 GGUF 导出。")


## 7. 双模型问答前端

左边是此次续训起点，右边是本轮按章续训后的模型。
每行一题，最多 12 道。支持连续对话和保存问答至 Drive。


## 7.1. 加载两份 LoRA 的推理接口

In [ ]:
import select
verify_run(persona_output, "distill_sft")
before_report = verify_run(initial_snapshot, INITIAL_TASK)
after_report = verify_run(persona_output, "distill_sft")
comparison_models = {
    "before": {"label": "继续训练前 · " + INITIAL_RUN, "path": str(initial_snapshot / "adapter"), "sha256": before_report["adapter_sha256"]},
    "current": {"label": "纠错续训后 · " + RUN_ID, "path": str(persona_output / "adapter"), "sha256": after_report["adapter_sha256"]},
}

COMPARE_WORKER = r'''
from contextlib import redirect_stdout
import json
from pathlib import Path
import sys

with redirect_stdout(sys.stderr):
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from yuki_model.persona import load_character

base = Path(sys.argv[1])
character = Path(sys.argv[2])
model = tokenizer = system = None

def reply(item):
    global model, tokenizer, system
    if model is None:
        if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
            raise RuntimeError("对比需要支持 BF16 的 CUDA GPU。")
        torch.cuda.set_device(0)
        system, _ = load_character(character, "training")
        tokenizer = AutoTokenizer.from_pretrained(base, local_files_only=True, trust_remote_code=False)
        backbone = AutoModelForCausalLM.from_pretrained(
            base, torch_dtype=torch.bfloat16, local_files_only=True, trust_remote_code=False,
        ).to("cuda:0")
        model = PeftModel.from_pretrained(backbone, item["path"], adapter_name=item["key"],
                                         is_trainable=False, local_files_only=True)
    elif item["key"] not in model.peft_config:
        model.load_adapter(item["path"], adapter_name=item["key"], is_trainable=False, local_files_only=True)
    model.set_adapter(item["key"])
    model.eval()
    messages = [{"role": "system", "content": system}, *item["messages"]]
    while True:
        encoded = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
        )
        length = encoded["input_ids"].shape[-1]
        if length <= 8192 - 192:
            break
        if len(messages) <= 2:
            raise ValueError("单题太长，请缩短后再试。")
        del messages[1:3]
    encoded = encoded.to("cuda:0")
    with torch.inference_mode():
        tokens = model.generate(**encoded, max_new_tokens=192, do_sample=False,
                                pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(tokens[0, length:], skip_special_tokens=True).strip()

for line in sys.stdin:
    try:
        with redirect_stdout(sys.stderr):
            request = json.loads(line)
            answers = [reply(item) for item in request]
        response = {"answers": answers}
    except Exception as error:
        response = {"error": f"{type(error).__name__}: {error}"}
    print(json.dumps(response, ensure_ascii=False), flush=True)
'''

if "stop_comparison_worker" in globals():
    stop_comparison_worker()
comparison_worker = None
comparison_log = drive_runs / "comparison-runtime.log"

def stop_comparison_worker():
    global comparison_worker
    worker = comparison_worker
    comparison_worker = None
    if worker is None:
        return
    if worker.poll() is None:
        worker.terminate()
        try:
            worker.wait(timeout=5)
        except subprocess.TimeoutExpired:
            worker.kill()
            worker.wait()
    worker.stdin.close()
    worker.stdout.close()

def compare_request(items):
    global comparison_worker
    if comparison_worker is None or comparison_worker.poll() is not None:
        stop_comparison_worker()
        with comparison_log.open("a", encoding="utf-8") as log:
            comparison_worker = subprocess.Popen(
                [str(python), "-u", "-c", COMPARE_WORKER, str(base), str(repo / "characters/yukino")],
                cwd=service, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=log,
                text=True, encoding="utf-8", bufsize=1,
                env=dict(os.environ, USE_TF="0", USE_TORCH="1"),
            )
    worker = comparison_worker
    try:
        worker.stdin.write(json.dumps(items, ensure_ascii=False) + "\n")
        worker.stdin.flush()
        if not select.select([worker.stdout], [], [], 300)[0]:
            raise TimeoutError("推理等待超过 5 分钟。")
        line = worker.stdout.readline()
        if not line:
            raise RuntimeError(f"推理进程退出，日志见 {comparison_log}")
        response = json.loads(line)
    except (OSError, ValueError, RuntimeError) as error:
        stop_comparison_worker()
        raise RuntimeError(f"{error}。可再次点击比较，日志见 {comparison_log}") from error
    if "error" in response:
        raise RuntimeError(response["error"])
    return response["answers"]

print("可选模型", "、".join(item["label"] for item in comparison_models.values()))


## 7.2. 打开批量与连续问答界面

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

import ipywidgets as widgets
from IPython.display import display

if "compare_request" not in globals():
    raise RuntimeError("请先运行 7.1。")

model_options = [(item["label"], key) for key, item in comparison_models.items()]
left = widgets.Dropdown(options=model_options, value="before",
                        description="左栏")
right = widgets.Dropdown(options=model_options, value="current", description="右栏")
mode = widgets.ToggleButtons(options=[("每题独立", "single"), ("连续对话", "chat")])
questions = widgets.Textarea(placeholder="每行一个问题，最多 12 行", layout=widgets.Layout(width="100%", height="108px"))
ask = widgets.Button(description="比较回答", button_style="primary")
clear = widgets.Button(description="清空对话")
release_gpu = widgets.Button(description="释放模型")
save_results = widgets.Checkbox(value=False, description="保存到 Drive", indent=False)
status = widgets.Label(value="请输入问题，首次提问时加载模型。")
answers = [widgets.Textarea(disabled=True, layout=widgets.Layout(width="100%", height="360px")) for _ in range(2)]
title = [widgets.Label(), widgets.Label()]
history = [[], []]
saved_file = drive_runs / "comparison" / f"{datetime.now(timezone.utc):%Y%m%d-%H%M%S}-{uuid4().hex[:6]}.jsonl"

def reset_comparison(_=None):
    history[0].clear()
    history[1].clear()
    for index, key in enumerate((left.value, right.value)):
        title[index].value = comparison_models[key]["label"]
        answers[index].value = ""
    status.value = "已清空对话。"

def release_comparison(_):
    stop_comparison_worker()
    status.value = "已释放模型，再次提问会重新加载。"

def submit_comparison(_):
    selected = (left.value, right.value)
    batch = [line.strip() for line in questions.value.splitlines() if line.strip()]
    if selected[0] == selected[1]:
        status.value = "请选择两个不同的模型。"
        return
    if not 1 <= len(batch) <= 12 or any(len(question) > 3000 for question in batch):
        status.value = "每次输入 1–12 道题，每题最多 3,000 字。"
        return
    controls = (left, right, mode, questions, ask, clear, release_gpu, save_results)
    for control in controls:
        control.disabled = True
    for box in answers:
        box.value = ""
    try:
        for number, question in enumerate(batch, 1):
            status.value = f"正在回答第 {number}/{len(batch)} 题。"
            user = {"role": "user", "content": question}
            items = [
                {"key": key, "path": comparison_models[key]["path"],
                 "messages": [*(history[index] if mode.value == "chat" else []), user]}
                for index, key in enumerate(selected)
            ]
            pair = compare_request(items)
            for index, answer in enumerate(pair):
                block = f"{number}. {question}\n\n{answer or '（模型未返回文字）'}"
                answers[index].value += ("\n\n" if answers[index].value else "") + block
                if mode.value == "chat":
                    history[index][:] = [*history[index], user, {"role": "assistant", "content": answer}][-8:]
            if save_results.value:
                saved_file.parent.mkdir(parents=True, exist_ok=True)
                record = {
                    "time": datetime.now(timezone.utc).isoformat(),
                    "mode": mode.value, "question": question,
                    "answers": [
                        {"model": comparison_models[key]["label"],
                         "adapter_sha256": comparison_models[key]["sha256"], "answer": answer}
                        for key, answer in zip(selected, pair)
                    ],
                }
                with saved_file.open("a", encoding="utf-8") as stream:
                    stream.write(json.dumps(record, ensure_ascii=False) + "\n")
        status.value = f"已回答 {len(batch)} 道题。" + (f" 已保存：{saved_file.name}" if save_results.value else "")
    except (OSError, RuntimeError, ValueError) as error:
        status.value = f"第 {number} 题出错：{error}"
    finally:
        for control in controls:
            control.disabled = False

for control in (left, right, mode):
    control.observe(reset_comparison, names="value")
ask.on_click(submit_comparison)
clear.on_click(reset_comparison)
release_gpu.on_click(release_comparison)
reset_comparison()
status.value = "请输入问题，首次提问时加载模型。"
columns = [widgets.VBox([title[index], answers[index]], layout=widgets.Layout(width="49%"))
           for index in range(2)]
display(widgets.VBox([
    widgets.HBox([left, right]), mode, questions,
    widgets.HBox([ask, clear, release_gpu, save_results]), status,
    widgets.HBox(columns, layout=widgets.Layout(width="100%", justify_content="space-between")),
]))


## 文件位置与恢复

`MyDrive/Yukino-Companion/retrain/epub-a100-v4-chapters/`：

- `corpus/`：章节、分段和提取清单；`teacher/`：逐段生成记录、问答及进度。
- `initial/`：起始 LoRA；`prepared/`：训练、验证、测试和数据清单。
- `distill-01/`：完整断点、最佳 `adapter/`、训练报告。
- `evaluation/`：同题对比；`distill-01-gguf/`：可选导出的离线模型。

断线后保持 `RUN_ID` 不变，重新运行 1、2、3、3.1，继续没有完成的步骤。
